In [1]:
import ast
import os
import pandas as pd
from tqdm import tqdm
import numpy as np

def extract_name(x):
    try:
        genus_dict = ast.literal_eval(x)
        return genus_dict.get('name', None)
    except:
        return None

os.chdir('/active-data/analysis_results/chr_pla')
all_data = pd.read_csv('genome_chr-pla_statistics.csv')
all_data['genus_clean'] = all_data['genus'].apply(extract_name)
counts = all_data['genus_clean'].value_counts()
keep_genus = counts[counts >= 400].index.to_list()
print(keep_genus)

['Escherichia', 'Klebsiella', 'Staphylococcus', 'Pseudomonas', 'Bacillus', 'Salmonella', 'Streptococcus', 'Streptomyces', 'Acinetobacter', 'Enterococcus', 'Bordetella', 'Enterobacter', 'Xanthomonas', 'Campylobacter', 'Vibrio', 'Mycobacterium', 'Corynebacterium', 'Burkholderia', 'Listeria', 'Citrobacter', 'Helicobacter']


In [2]:
def max_dict(dic):
    max_num = None
    for key in dic:
        try:
            int(max_num)
        except:
            max_num = dic[key]
        if dic[key] >= max_num:
            max_key = key
            max_num = dic[key]
    return max_key, max_num

def replicon_pd(org_data_n):
    MS_replicon = pd.DataFrame(columns=['id', 'size']) # Maximum-size replicon
    NMS_replicon = pd.DataFrame(columns=['id', 'size']) # Non-maximum-size replicon
    for i in org_data_n.index:
        acc_n = org_data_n['accession'][i]
        chr_data = ast.literal_eval(org_data_n['chromosome contigs'][i])
        pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
        merge_data = chr_data | pla_data
        max_key, max_num = max_dict(merge_data)
        MS_replicon = pd.concat([MS_replicon, pd.DataFrame([{'id': acc_n+'-'+max_key, 'size': max_num}])], ignore_index=True)
        for id_n in chr_data:
            if acc_n+'-'+id_n in MS_replicon['id'].values:
                continue
            else:
                NMS_replicon = pd.concat([NMS_replicon, pd.DataFrame([{'id': acc_n+'-'+id_n, 'size': chr_data[id_n]}])], ignore_index=True)
        for id_n in pla_data:
            if acc_n+'-'+id_n in MS_replicon['id'].values:
                continue
            else:
                NMS_replicon = pd.concat([NMS_replicon, pd.DataFrame([{'id': acc_n+'-'+id_n, 'size': pla_data[id_n]}])], ignore_index=True)

    return MS_replicon, NMS_replicon

def process_replicon_data(fraction_data, threshold, label):
    df = fraction_data[fraction_data[f'average plasmid fraction-{label}'] < threshold].copy()

    df['log10_size'] = np.log10(df['size'])
    df_sorted = df.sort_values('log10_size', ascending=False).reset_index(drop=True)

    if len(df_sorted) >= 2:
        log_vals = df_sorted['log10_size'].values
        diffs = np.diff(log_vals)
        abs_diffs = np.abs(diffs)
        
        first_jump = np.where(abs_diffs > 0.25)[0]
        
        if len(first_jump) > 0 and df_sorted.iloc[first_jump[0]]['size'] > 1e5:
            pos = first_jump[0]
            size1 = df_sorted.iloc[pos]['size']
            size2 = df_sorted.iloc[pos + 1]['size']
            return (size1 + size2) / 2

    df_mid = df[(df['size'] >= 1e5) & (df['size'] <= 1e6)].copy()
    
    if len(df_mid) >= 2:
        df_mid['log10_size'] = np.log10(df_mid['size'])
        df_mid_sorted = df_mid.sort_values('log10_size', ascending=False).reset_index(drop=True)
        
        mid_logs = df_mid_sorted['log10_size'].values
        mid_diffs = np.diff(mid_logs)
        mid_abs_diffs = np.abs(mid_diffs)
        
        max_pos = np.argmax(mid_abs_diffs)
        s1 = df_mid_sorted.iloc[max_pos]['size']
        s2 = df_mid_sorted.iloc[max_pos + 1]['size']
        return (s1 + s2) / 2

    min_size = df['size'].min()
    return min_size * 0.5

def check_ms_replicon_conditions(fraction_data, size_split, threshold, label):
    ms_replicon_rows = fraction_data[fraction_data['ms-label'] == 'MS_replicon'].copy()
    
    if ms_replicon_rows.empty:
        return True, pd.DataFrame()

    condition = (
        (ms_replicon_rows['size'] > size_split) &
        (ms_replicon_rows[f'average plasmid fraction-{label}'] < threshold)
    )
    all_satisfied = condition.all()
    unsatisfied_rows = ms_replicon_rows[~condition]
    
    return all_satisfied, unsatisfied_rows

def self_bitscore(genus_name, org_data_n, NMS_replicon, folder):
    labels = ['original', 'pident_90', 'pident_95']
    fraction_data = pd.DataFrame()
    with tqdm(total = len(org_data_n), desc=f'{genus_name}({len(org_data_n)})', leave=True, ncols=100, unit='B', unit_scale=True) as pbar:
        NMS_replicon_list = list(NMS_replicon['id'])
        file_folder = f'/active-data/analysis_results/chr_pla/genus/cor-pla_fraction_records/{genus_name}'
        for i in org_data_n.index:
            acc_n = org_data_n['accession'][i]
            chr_data = ast.literal_eval(org_data_n['chromosome contigs'][i])
            pla_data = ast.literal_eval(org_data_n['plasmid contigs'][i])
            temp_data = {}
            bitscore_csv = pd.read_csv(f'{file_folder}/{acc_n}/replicon_bitscore.csv', index_col = 0)
            for item in chr_data:
                temp_data['organism'] = extract_name(org_data_n['species'][i])
                temp_data['accession'] = acc_n+'-'+item
                temp_data['size'] = chr_data[item]
                file = open(f'{file_folder}/{acc_n}/{item}.txt', 'r')
                for line in file:
                    list_a = ast.literal_eval(line)
                    for item_n in list_a:
                        temp_data[item_n] = list_a[item_n]
                        break
                file.close()
                for label in labels:
                    temp_data[f'self_bitscore-{label}'] = bitscore_csv[f'{item}-{label}'][temp_data['accession']]
                temp_data['cp-label'] = 'chromosome'
                if acc_n+'-'+item in NMS_replicon_list:
                    temp_data['ms-label'] = 'NMS_replicon'
                else:
                    temp_data['ms-label'] = 'MS_replicon'
                fraction_data = pd.concat([fraction_data, pd.DataFrame([temp_data])], ignore_index=True)
                temp_data = {}
            for item in pla_data:
                temp_data['organism'] = extract_name(org_data_n['species'][i])
                temp_data['accession'] = acc_n+'-'+item
                temp_data['size'] = pla_data[item]
                file = open(f'{file_folder}/{acc_n}/{item}.txt', 'r')
                for line in file:
                    list_a = ast.literal_eval(line)
                    for item_n in list_a:
                        temp_data[item_n] = list_a[item_n]
                        break
                file.close()
                for label in labels:
                    try: temp_data[f'self_bitscore-{label}'] = bitscore_csv[f'{item}-{label}'][temp_data['accession']]
                    except: pass
                temp_data['cp-label'] = 'plasmid'
                if acc_n+'-'+item in NMS_replicon_list:
                    temp_data['ms-label'] = 'NMS_replicon'
                else:
                    temp_data['ms-label'] = 'MS_replicon'
                fraction_data = pd.concat([fraction_data, pd.DataFrame([temp_data])], ignore_index=True)
                temp_data = {}
            pbar.update(1)

    threshold = 0.3
    for label in labels:
        size_split = process_replicon_data(fraction_data, threshold, label)
        
        col_name = f'category-{label}'
        frac_col = f'average plasmid fraction-{label}'
        
        conditions = [
            (fraction_data['size'] >= size_split) & (fraction_data[frac_col] < threshold),
            (fraction_data['size'] < size_split) & (fraction_data[frac_col] < threshold),
            fraction_data[frac_col] >= threshold
        ]
        choices = [
            'typical chromosome',
            'intermediate replicon',
            'typical plasmid'
        ]
        fraction_data[col_name] = np.select(conditions, choices, default='unknown')
        all_ok, bad_rows = check_ms_replicon_conditions(fraction_data, size_split, threshold, label)
        print(f'{genus_name}-{label}', all_ok)
        
    os.chdir(folder)
    fraction_data.to_csv('replicon-plasmid_fraction-self_bitscore_statistics.csv', index=False)

In [3]:
for genus_name in keep_genus:
    org_data_n = all_data[all_data['genus_clean'].str.contains(genus_name, na=False)].reset_index(drop=True)
    folder = f'/active-data/analysis_results/chr_pla/genus/statistics_records/{genus_name}'
    if not os.path.exists(folder):
        os.makedirs(folder)

    MS_replicon, NMS_replicon = replicon_pd(org_data_n)    
    self_bitscore(genus_name, org_data_n, NMS_replicon, folder)

Escherichia(4204): 100%|████████████████████████████████████████| 4.20k/4.20k [15:04<00:00, 4.65B/s]


Escherichia-original True
Escherichia-pident_90 True
Escherichia-pident_95 True


Klebsiella(3554): 100%|█████████████████████████████████████████| 3.55k/3.55k [14:16<00:00, 4.15B/s]


Klebsiella-original True
Klebsiella-pident_90 True
Klebsiella-pident_95 True


Staphylococcus(2423): 100%|█████████████████████████████████████| 2.42k/2.42k [03:24<00:00, 11.8B/s]


...

Listeria-original True
Listeria-pident_90 True
Listeria-pident_95 True


Citrobacter(420): 100%|█████████████████████████████████████████████| 420/420 [01:08<00:00, 6.16B/s]


Citrobacter-original True
Citrobacter-pident_90 True
Citrobacter-pident_95 True


Helicobacter(416): 100%|████████████████████████████████████████████| 416/416 [00:17<00:00, 23.7B/s]

Helicobacter-original True
Helicobacter-pident_90 True
Helicobacter-pident_95 True
